# Exportar el modelo real desde Google Colab

Este notebook **no entrena ni vuelve a evaluar**. Recupera el artefacto `logistic_regression_classifier.joblib` generado por la Actividad 6, conserva el encoder MiniLM y las 77 etiquetas, registra versiones y prepara el paquete para Docker/GitHub.

Antes de ejecutar: sube y descomprime el ZIP `Asistente_Tickets_Despliegue.zip` en `/content/`, y asegÃºrate de que el clasificador exista en tu Drive dentro de `Actividad5/modelos/`. Si no existe, regresa al notebook de Actividad 6 y ejecuta Ãºnicamente las celdas de carga/guardado de modelos que correspondan; no ejecutes GridSearch ni la evaluaciÃ³n completa.

El artefacto `.joblib` debe proceder de tu propio experimento. No cargues modelos serializados de fuentes desconocidas.

In [ ]:
from pathlib import Path
import os, json, shutil, sys, platform, subprocess, zipfile
from datetime import datetime, timezone

# Localiza la carpeta del paquete descomprimido.
candidates = [Path('/content/Asistente_Tickets_Despliegue'), Path.cwd()]
DEPLOY_ROOT = next((p.resolve() for p in candidates if (p / 'app' / 'main.py').exists()), None)
if DEPLOY_ROOT is None:
    raise FileNotFoundError('No encuentro /content/Asistente_Tickets_Despliegue. Sube y descomprime primero el ZIP del proyecto.')
print('Proyecto:', DEPLOY_ROOT)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_SEARCH_ROOT = Path('/content/drive/MyDrive')
if not DRIVE_SEARCH_ROOT.exists():
    DRIVE_SEARCH_ROOT = Path('/content/drive/MyDrive')
matches = list(DRIVE_SEARCH_ROOT.glob('**/Actividad5/modelos/logistic_regression_classifier.joblib'))
if not matches:
    raise FileNotFoundError('No se encontrÃ³ Actividad5/modelos/logistic_regression_classifier.joblib. No se entrenarÃ¡ nada desde este notebook. Revisa el notebook Actividad 6 y localiza el artefacto guardado.')
if len(matches) > 1:
    print('Se encontraron varios artefactos; se utilizarÃ¡ el primero:', matches[0])
SOURCE_MODEL = matches[0]
ACTIVITY5_ROOT = SOURCE_MODEL.parents[1]
print('Modelo fuente:', SOURCE_MODEL)
print('RaÃ­z Actividad 5:', ACTIVITY5_ROOT)

In [ ]:
# Exportar el clasificador y el diccionario canÃ³nico de 77 intenciones.
import joblib, numpy as np, pandas as pd

bundle = joblib.load(SOURCE_MODEL)
if isinstance(bundle, dict) and 'classifier' in bundle:
    classifier = bundle['classifier']
    encoder_id = bundle.get('encoder_id') or 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
    label_names = bundle.get('label_names')
else:
    classifier = bundle
    encoder_id = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
    label_names = None

if label_names is None or len(label_names) != 77:
    dictionary_path = ACTIVITY5_ROOT / 'datos' / 'diccionario_intenciones.csv'
    if not dictionary_path.exists():
        raise FileNotFoundError('El bundle no tiene label_names y tampoco se encontrÃ³ datos/diccionario_intenciones.csv en Actividad5.')
    dictionary = pd.read_csv(dictionary_path)
    label_names = [None] * 77
    for row in dictionary.itertuples(index=False):
        label_names[int(row.label_id)] = str(row.intencion)

if len(label_names) != 77 or any(name is None for name in label_names):
    raise ValueError('El catÃ¡logo debe contener exactamente 77 nombres, ordenados por ID 0-76.')
classes = [int(x) for x in classifier.classes_]
if len(classes) != 77 or sorted(classes) != list(range(77)):
    raise ValueError(f'El clasificador no contiene las 77 clases esperadas: {classes[:10]}...')

models_dir = DEPLOY_ROOT / 'models'
models_dir.mkdir(exist_ok=True)
deployment_bundle = {'classifier': classifier, 'encoder_id': encoder_id, 'label_names': [str(x) for x in label_names]}
target_model = models_dir / 'logistic_regression_classifier.joblib'
joblib.dump(deployment_bundle, target_model, compress=3)
labels_json = {str(i): str(label_names[i]) for i in range(77)}
(models_dir / 'intent_labels.json').write_text(json.dumps(labels_json, ensure_ascii=False, indent=2), encoding='utf-8')
print('Clasificador exportado:', target_model, f'({target_model.stat().st_size:,} bytes)')
print('Etiquetas:', len(labels_json))
print('Encoder:', encoder_id)

In [ ]:
# Crear manifiesto y lock con las versiones esenciales del entorno que generÃ³ el clasificador.
import importlib.metadata as metadata

# FastAPI/uvicorn se instalan si no estÃ¡n presentes; no afecta el entrenamiento ni el modelo.
for module, package in [('fastapi', 'fastapi'), ('uvicorn', 'uvicorn[standard]')]:
    try:
        metadata.version(module)
    except metadata.PackageNotFoundError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

dist_names = ['fastapi', 'uvicorn', 'sentence-transformers', 'scikit-learn', 'joblib', 'numpy', 'torch']
versions = {name: metadata.version(name) for name in dist_names}
encoder_revision = None
try:
    from huggingface_hub import model_info
    encoder_revision = model_info(encoder_id).sha
except Exception as exc:
    print('No se pudo fijar el commit del encoder; se conserva el ID pÃºblico:', exc)

manifest = {
    'model_name': 'logistic_regression',
    'exported_at_utc': datetime.now(timezone.utc).isoformat(),
    'source_artifact': str(SOURCE_MODEL),
    'encoder_id': encoder_id,
    'encoder_revision': encoder_revision,
    'number_of_intents': 77,
    'embedding_normalized': True,
    'confidence_threshold_initial': 0.50,
    'threshold_status': 'exploratory_not_calibrated_not_approved_for_production',
    'priority_automation': False,
    'python_version': platform.python_version(),
    'versions': versions
}
(models_dir / 'model_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

# Para el archivo lock se utiliza PyTorch sin sufijo CUDA: el contenedor se ejecutarÃ¡ en CPU.
torch_cpu_version = versions['torch'].split('+')[0]
lock_lines = [
    '--extra-index-url https://download.pytorch.org/whl/cpu',
    f"fastapi=={versions['fastapi']}",
    f"uvicorn[standard]=={versions['uvicorn']}",
    f"sentence-transformers=={versions['sentence-transformers']}",
    f"scikit-learn=={versions['scikit-learn']}",
    f"joblib=={versions['joblib']}",
    f"numpy=={versions['numpy']}",
    f"torch=={torch_cpu_version}",
]
(DEPLOY_ROOT / 'requirements-runtime.lock').write_text('\n'.join(lock_lines) + '\n', encoding='utf-8')
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print('Lock creado:', DEPLOY_ROOT / 'requirements-runtime.lock')

In [ ]:
# VerificaciÃ³n y paquete listo para transferir a tu computadora.
required = [models_dir / 'logistic_regression_classifier.joblib', models_dir / 'intent_labels.json', models_dir / 'model_manifest.json', DEPLOY_ROOT / 'requirements-runtime.lock']
missing = [str(p) for p in required if not p.is_file() or p.stat().st_size == 0]
if missing:
    raise FileNotFoundError('Faltan archivos: ' + ', '.join(missing))

output_zip = Path('/content/Asistente_Tickets_Despliegue_LISTO.zip')
if output_zip.exists(): output_zip.unlink()
with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for path in DEPLOY_ROOT.rglob('*'):
        if path.is_file() and not any(part in {'.git', '__pycache__', '.pytest_cache', '.venv'} for part in path.parts):
            zf.write(path, path.relative_to(DEPLOY_ROOT.parent))
print('Paquete listo:', output_zip, f'({output_zip.stat().st_size / (1024**2):.1f} MB)')

from google.colab import files
files.download(str(output_zip))